In [15]:
import gurobipy as gp
from gurobipy import GRB
import math

 #importo la libreria che mi permette di creare un modello matematico,aggiungere variabili , aggiungere vincoli, definire la FO , lanciare l'ottimizzazione e leggerne i risultati
 #la classe GRB , che è una parte specifica della libreria di gurobi , contiene tutte le costanti usate per definire il tipo di variabile ed il tipo di ottimizzazione (binaria,intera,continua , max\min)
 #import math importa funzioni matematiche o meglio la libreria matematica standard di python

 # DEFINISCO I VARI INSIEMI

customers = [1,2,3,4,5] #sotto-insieme dei nodi clienti 1-5
lockers = [6,7] #sotto-insieme dei nodi lockers 6-7
nodes = [0] + customers + lockers #insieme totale dei nodi + il nodo 0 che funge da deposito

 # DEFINISCO LE COORDINATE DI OGNI NODO, CALCOLO LA DISTANZA EUCLIDEA PER OGNI COPPIA DI NODI (i,j)

coords = {0: (10,10), 1: (2,8), 2: (10,2), 3: (15,10), 4: (5,15), 5: (18,5), 6: (7,7), 7: (13,13)}
dist = { (i,j): math.sqrt((coords[i][0]-coords[j][0])**2 + (coords[i][1]-coords[j][1])**2) for i in nodes for j in nodes}

 # DEFINISCO IL RAGGIO DI AMMISSIBILITA'\COPERTURA DEL LOCKER , ogni locker (6-7) ha un raggio massimo di copertura pari a 10
R = {
    6: 10,
    7: 10}
 # COMPATIBILITA' CLIENTE-LOCKER , la matrice a[i,b] indica se il cliente i può usare quel locker b - vincolo di feasibility geografica
a = {}
 
for i in customers:
    for b in lockers:
        if dist[i,b] <= R[b]:
            a[i,b] = 1
        else:
            a[i,b] = 0
 
# IMPOSTAZIONE DEI PARAMETRI FONDAMENTALI

Q = 3             #capacità del veicolo , max 3 pacchi trasportati per veicolo
v_b = 2            #capacità del locker , max 2 clienti serviti dal locker
alpha1 = 1
alpha2 = 0.1
#alpha1 ed alpha2 sono i coefficienti di peso della funzione obiettivo,tarati in modo tale che il trasporto (domicilio) 
#pesi molto più dell'accesso al locker da parte del cliente
 
# CREO IL MODELLO CHE CONTERRA' VARIABILI , FO E VINCOLI

model = gp.Model("VRPL_Optimization")

 # DEFINISCO LE VARIABILI DECISIONALI

x = model.addVars(nodes, nodes, vtype=GRB.BINARY, name="x")        #1 se il veicolo percorre l'arco (i,j) ,0 altrimenti
y = model.addVars(customers, lockers, vtype=GRB.BINARY, name="y")  #1 se il cliente i è assegnato al locker b ,0 altrimenti
f = model.addVars(nodes, nodes, vtype=GRB.CONTINUOUS, name="f")    #variabile di flusso (continua) ,rappresenta il carico trasportato dal veicolo lungo l'arco (i,j)
 
# DEFINISCO LA FUNZIONE OBIETTIVO

obj = alpha1 * gp.quicksum(dist[i,j] * x[i,j] for i in nodes for j in nodes if i != j) + \
      alpha2 * gp.quicksum(dist[i,b] * y[i,b] for i in customers for b in lockers)
model.setObjective(obj, GRB.MINIMIZE)
 
# il primo termine minimizza la distanza percorsa dal veicolo per effettuare la consegna a domicilio - costo del trasporto 
#(include costi relativi a possibili usure dei mezzi, costi relativi al consumo di carburante e costi relativi al tempo impiegato dagli autisti)
# il secondo termine minimizza la distanza percorsa dal cliente per arrivare al locker assegnato - costo di accesso dei clienti al locker
# if i != j evita di calcolare la distanza di un nodo con se stesso che sarebbe uguale a 0

 
# DEFINISCO I VINCOLI DI PERCORSO - ROUTING

#questo vincolo garantisce la continuità della rotta , la somma degli archi entranti in un nodo deve essere uguale alla somma degli archi uscenti 
#dunque se il veicolo visita un nodo , deve anche lasciarlo evitando nodi isolati o percorsi isolati
for i in nodes:
    model.addConstr(gp.quicksum(x[j,i] for j in nodes if j != i) == 
                    gp.quicksum(x[i,j] for j in nodes if i != j))
 
#questo vincolo garantisce che ogni cliente sia servito esattamente una volta o dal veicolo o da un locker,la somma degli archi entranti (xji) nel cliente rappresenta la consegna 
#a domicilio , mentre la somma delle variabili yᵢᵇ rappresenta la consegna via locker quindi impongo che la loro somma sia pari a 1 
for i in customers:
    model.addConstr(gp.quicksum(x[j,i] for j in nodes if j != i ) +
                    gp.quicksum(y[i,b] for b in lockers) ==1)
    
#questo vincolo garantisce la compatibilità geografica cliente-locker, la matrice a[i,b] indica se il cliente i si trova entro il raggio 
#di copertura del locker b, impongo poi la disuguaglianza cosicchè il cliente possa essere assegnato ad un locker solo se è effettivamente raggiungibile
for i in customers:
    for b in lockers:
        model.addConstr(y[i,b] <= a[i,b])
 
#questo vincolo garantisce che ogni locker non superi la propria capacità fisica , la variabile yᵢᵇ indica se il cliente i è assegnato al locker b; 
#la loro somma rappresenta il numero totale di clienti serviti da quel locker dunque impongo che questa somma sia ≤ v_b, cioè la capacità massima 
#del locker.(ogni locker può servire al massimo 2 clienti)
for b in lockers:
    model.addConstr(gp.quicksum(y[i, b] for i in customers) <= v_b, 
                    name=f"Capacita_Locker_{b}")
 

#questo vincolo collega l’assegnamento dei clienti ai locker con il routing del veicolo, quindi se un cliente è assegnato a un locker, la 
#variabile yᵢᵇ vale 1 ed in tal caso, il modello impone che almeno un arco entrante nel locker sia attivo, cioè il veicolo deve visitarlo.
for i in customers:
    for b in lockers:
        model.addConstr(gp.quicksum(x[j,b] for j in nodes if j != b) >= y[i,b] )
 

#questo vincolo limita il numero di veicoli che possono partire dal deposito (al max 2) , la somma delle variabili x0j 
#(1 se il veicolo percorre l'arco dal deposito al nodo j) rappresenta il numero di rotte che iniziano dal deposito.
model.addConstr(gp.quicksum(x[0,j] for j in nodes if j != 0) <= 2)
 
#questo vincolo limita il numero di veicoli che possono rientrare al deposito (al max 2), la somma delle variabili xi0 
#(1 se il veicolo percorre l'arco dal nodo i al deposito) rappresenta il numero di rotte che terminano al deposito.
model.addConstr(gp.quicksum(x[i,0] for i in nodes if i != 0) <= 2)
 
# DEFINISCO I VINCOLI DI FLUSSO (OCF)

q = {i: 1 for i in customers}  #definisco la domanda dei clienti come un'unità di carico ciascuno , cioè ogni cliente richiede un pacco
 
model.addConstr(gp.quicksum(f[0,i] for i in nodes if i != 0) == 5) #quantità totale di carico che esce dal deposito , la somma del flusso in uscita 
#dal deposito deve essere pari alla somma delle domande dei clienti cioè pari a 5

#questo vincolo è relativo alla conservazione del flusso sul nodo cliente ,la somma del flusso entrante deve essere uguale al flusso uscente 
#più la domanda effettiva del cliente.Se il cliente è servito a domicilio, la domanda vale 1 e il carico diminuisce di una unità ,
#se invece il cliente è assegnato a un locker, la domanda vale 0 e il carico non diminuisce sul nodo cliente
for i in customers:
    model.addConstr(gp.quicksum(f[j,i] for j in nodes if j != i) == 
                    gp.quicksum(f[i,j] for j in nodes if i != j) + 
                    1 * (1 - gp.quicksum(y[i,b] for b in lockers)))
 
#questo vincolo è relativo alla conservazione del flusso sul nodo locler , la somma del flusso entrante deve essere pari alla somma del flusso uscente 
#più la quantità di merce che il veicolo deve scaricare al locker , pari alla somma delle domande dei clienti assegnati a quel locker
for b in lockers:
    model.addConstr(gp.quicksum(f[j,b] for j in nodes if j != b) == 
                    gp.quicksum(f[b,j] for j in nodes if b != j) + 
                    gp.quicksum(q[i] * y[i,b] for i in customers))
 
 
#questo vincolo lega la variabile di flusso alla variabile di routing ,se l'arco (i,j) non è percorso dal veicolo  (xij=0) allora il flusso su quell'arco deve esser nullo
#se l'arco (i,j) è percorso (xij=1) allora il flusso su quell'arco deve rispettare la capacità del veicolo
for i in nodes:
    for j in nodes:
        model.addConstr(f[i,j] <= Q * x[i,j])
 
#questo vincolo elimina gli archi che partono e arrivano sullo stesso nodo
for i in nodes:
    model.addConstr(x[i,i] == 0)  #impongo x[i,i] = 0 per evitare che il veicolo percorra archi non realistici
    model.addConstr(f[i,i] == 0)  #impongo f[i,i] = 0 per impedire che il flusso  circoli su archi inesistenti
 
# OTTIMIZZAZIONE ED ANALISI DELLA SOLUZIONE
 
model.optimize()   #gurobi costruisce il modello, applica presolve, avvia il Branch&Bound e cerca la soluzione ottima
 
# Stampa dei risultati
if model.status == GRB.OPTIMAL:
    print(f"\nCosto Ottimo Totale: {model.objVal}")       #controlla che Gurobi abbia trovato una soluzione ottima,se sì stampa il valore della funzione obiettivo, cioè il costo totale della soluzione.
    for i, j in x.keys():
        if x[i,j].x > 0.5:      #scorre tutte le variabili xij , se xij=1 (o maggiore di 0.5 per tolleranza numerica) significa che il veicolo percorre l'arco (i,j)
            print(f"Percorso: {i} -> {j} (Carico trasportato: {f[i,j].x})") #stampa dunque l'arco percorso ed il carico trasportato su quell'arco
    for i, b in y.keys():      
        if y[i,b].x > 0.5:       #scorre tutte le variabili yᵢᵇ , se yᵢᵇ=1 (o maggiore di 0.5 per tolleranza numerica) significa che il cliente i è assegnato al locker b
            print(f"Il cliente {i} ritirerà il pacco al Locker {b}")     #stampa dunque che il cliente ritirerà il pacco al locker , serve per distinguere quali clienti sono serviti via locker e quali a domicilio
            for i in customers:
                print("\nAssegnamenti ai locker:")
 
for i in customers:
    for b in lockers:
        if y[i,b].X > 0.5:
            print(f"Cliente {i} assegnato al locker {b}")
 


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 3 3200U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 125 rows, 138 columns and 540 nonzeros
Model fingerprint: 0xfae1e2ec
Variable types: 64 continuous, 74 integer (74 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [4e-01, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]
Found heuristic solution: objective 48.3489124
Presolve removed 37 rows and 19 columns
Presolve time: 0.00s
Presolved: 88 rows, 119 columns, 457 nonzeros
Variable types: 56 continuous, 63 integer (63 binary)

Root relaxation: objective 2.290112e+01, 69 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Ti